# Path Bar

> Path display, breadcrumbs navigation, and optional path input for the file browser.

In [ ]:
#| default_exp components.path_bar

In [ ]:
#| export
from pathlib import Path
from typing import Any, List, Optional, Tuple

from fasthtml.common import Div, Span, Button, Ul, Li, A, Input, Form

# DaisyUI components
from cjm_fasthtml_daisyui.components.actions.button import btn, btn_colors, btn_sizes, btn_styles
from cjm_fasthtml_daisyui.components.data_input.text_input import text_input, text_input_sizes
from cjm_fasthtml_daisyui.components.navigation.breadcrumbs import breadcrumbs
from cjm_fasthtml_daisyui.utilities.semantic_colors import bg_dui, text_dui, border_dui

# Tailwind utilities
from cjm_fasthtml_tailwind.utilities.spacing import p, m
from cjm_fasthtml_tailwind.utilities.sizing import w, h, min_w
from cjm_fasthtml_tailwind.utilities.typography import font_size, font_family, truncate
from cjm_fasthtml_tailwind.utilities.borders import border, rounded
from cjm_fasthtml_tailwind.utilities.flexbox_and_grid import (
    flex_display, flex_direction, justify, items, gap, grow, shrink
)
from cjm_fasthtml_tailwind.utilities.layout import overflow
from cjm_fasthtml_tailwind.core.base import combine_classes

# Lucide icons
from cjm_fasthtml_lucide_icons.factory import lucide_icon

# Design system recipes (V11 icon-size roles)
from cjm_fasthtml_design_system.icons import icons

# Local imports
from cjm_fasthtml_file_browser.core.config import FileBrowserConfig
from cjm_fasthtml_file_browser.core.html_ids import FileBrowserHtmlIds
from cjm_fasthtml_file_browser.components.item import BROWSER_ICONS

## Path Parsing

Utilities for parsing paths into breadcrumb segments.

In [ ]:
#| export
def parse_path_segments(
    path: str,                    # Full path to parse
    separator: str = "/"          # Path separator
) -> List[Tuple[str, str]]:  # List of (name, full_path) tuples
    """Parse a path into breadcrumb segments."""
    p = Path(path)
    segments = []
    
    # Build segments from root to current
    parts = list(p.parts)
    for i, part in enumerate(parts):
        # Build full path up to this segment
        if i == 0 and part == separator:
            # Root
            full_path = separator
            name = separator
        else:
            full_path = str(Path(*parts[:i+1]))
            name = part
        segments.append((name, full_path))
    
    return segments

In [ ]:
# Test path parsing
segments = parse_path_segments("/home/user/Documents")
assert len(segments) == 4  # /, home, user, Documents
assert segments[0] == ("/", "/")
assert segments[1][0] == "home"
assert segments[-1][0] == "Documents"

# Test single segment
segments = parse_path_segments("/")
assert len(segments) == 1
assert segments[0] == ("/", "/")

print("Path parsing tests passed!")

Path parsing tests passed!


## Breadcrumbs

Renders the path as clickable breadcrumb links.

In [ ]:
#| export
def render_breadcrumbs(
    current_path: str,                      # Current directory path
    navigate_url: str,                      # URL for navigation
    hx_target: Optional[str] = None,        # HTMX target for swaps
    max_segments: int = 5,                  # Max segments to show (0 = all)
    breadcrumbs_id: Optional[str] = None,   # HTML ID for breadcrumbs
) -> Any:  # Breadcrumbs component
    """Render path as breadcrumb navigation."""
    segments = parse_path_segments(current_path)
    
    # Truncate if too many segments
    show_ellipsis = False
    if max_segments > 0 and len(segments) > max_segments:
        # Keep first and last segments, add ellipsis in between
        segments = [segments[0]] + segments[-(max_segments-1):]
        show_ellipsis = True
    
    # Build list items
    items_list = []
    
    for i, (name, full_path) in enumerate(segments):
        is_last = (i == len(segments) - 1)
        
        # Add ellipsis after first segment if truncated
        if show_ellipsis and i == 1:
            items_list.append(Li(Span("...", cls=str(text_dui.base_content.opacity(50)))))
        
        if is_last:
            # Current location - not a link
            display_name = name if name != "/" else "Root"
            items_list.append(Li(Span(display_name, cls=str(truncate))))
        else:
            # Clickable link
            display_name = name if name != "/" else "Root"
            link_attrs = {
                "hx_post": navigate_url,
                "hx_vals": f'{{"path": "{full_path}"}}',
                "hx_swap": "outerHTML",
                "cls": "cursor-pointer hover:underline"
            }
            if hx_target:
                link_attrs["hx_target"] = hx_target
            
            items_list.append(Li(Span(display_name, **link_attrs)))
    
    # Build breadcrumbs
    bc_attrs = {"cls": combine_classes(breadcrumbs, font_size.sm)}
    if breadcrumbs_id:
        bc_attrs["id"] = breadcrumbs_id
    
    return Div(
        Ul(*items_list),
        **bc_attrs
    )

In [ ]:
from fasthtml.common import to_xml

# Test breadcrumbs
bc = render_breadcrumbs("/home/user/Documents", "/navigate")
html = to_xml(bc)
assert "Root" in html  # / becomes Root
assert "home" in html
assert "user" in html
assert "Documents" in html
assert "breadcrumbs" in html

# Test with HTMX target
bc = render_breadcrumbs("/home/user", "/nav", hx_target="#browser")
html = to_xml(bc)
assert "hx-target" in html
assert "#browser" in html

print("Breadcrumbs tests passed!")

Breadcrumbs tests passed!


## Path Input

Optional text input for direct path entry.

In [ ]:
#| export
def render_path_input(
    current_path: str,                      # Current path to display
    navigate_url: str,                      # URL for navigation
    hx_target: Optional[str] = None,        # HTMX target for swaps
    input_id: Optional[str] = None,         # HTML ID for input
) -> Any:  # Path input component
    """Render a text input for direct path entry."""
    input_attrs = {
        "type": "text",
        "name": "path",
        "value": current_path,
        "cls": combine_classes(text_input, text_input_sizes.sm, w.full, font_family.mono),
        "placeholder": "Enter path...",
    }
    if input_id:
        input_attrs["id"] = input_id
    
    form_attrs = {
        "hx_post": navigate_url,
        "hx_swap": "outerHTML",
        "cls": combine_classes(flex_display, gap(2), w.full)
    }
    if hx_target:
        form_attrs["hx_target"] = hx_target
    
    return Form(
        Input(**input_attrs),
        Button(
            "Go",
            type="submit",
            cls=combine_classes(btn, btn_colors.primary, btn_sizes.sm)
        ),
        **form_attrs
    )

In [ ]:
# Test path input
path_input = render_path_input("/home/user", "/navigate")
html = to_xml(path_input)
assert "<form" in html.lower()
assert "<input" in html.lower()
assert "/home/user" in html
assert "Go" in html

print("Path input tests passed!")

Path input tests passed!


## Navigation Buttons

Quick navigation buttons (parent, home, refresh).

In [ ]:
#| export
def render_nav_buttons(
    parent_path: Optional[str],             # Parent directory path (None if at root)
    home_path: str,                         # Home directory path
    navigate_url: str,                      # URL for navigation
    refresh_url: Optional[str] = None,      # URL for refresh (if different)
    hx_target: Optional[str] = None,        # HTMX target for swaps
) -> Any:  # Navigation buttons component
    """Render quick navigation buttons."""
    ids = FileBrowserHtmlIds()
    buttons = []
    
    # Parent button
    parent_attrs = {
        "id": ids.BTN_PARENT,
        "cls": combine_classes(btn, btn_styles.ghost, btn_sizes.sm),
        "title": "Go to parent directory",
    }
    if parent_path:
        parent_attrs["hx_post"] = navigate_url
        parent_attrs["hx_vals"] = f'{{"path": "{parent_path}"}}'
        parent_attrs["hx_swap"] = "outerHTML"
        if hx_target:
            parent_attrs["hx_target"] = hx_target
    else:
        parent_attrs["disabled"] = True
        parent_attrs["cls"] = combine_classes(parent_attrs["cls"], "opacity-50")
    
    buttons.append(Button(
        lucide_icon(BROWSER_ICONS["parent"], size=icons.icon_button),
        **parent_attrs
    ))
    
    # Home button
    home_attrs = {
        "id": ids.BTN_HOME,
        "hx_post": navigate_url,
        "hx_vals": f'{{"path": "{home_path}"}}',
        "hx_swap": "outerHTML",
        "cls": combine_classes(btn, btn_styles.ghost, btn_sizes.sm),
        "title": "Go to home directory",
    }
    if hx_target:
        home_attrs["hx_target"] = hx_target
    
    buttons.append(Button(
        lucide_icon(BROWSER_ICONS["home"], size=icons.icon_button),
        **home_attrs
    ))
    
    # Refresh button
    refresh_attrs = {
        "id": ids.BTN_REFRESH,
        "hx_post": refresh_url or navigate_url,
        "hx_swap": "outerHTML",
        "cls": combine_classes(btn, btn_styles.ghost, btn_sizes.sm),
        "title": "Refresh",
    }
    if hx_target:
        refresh_attrs["hx_target"] = hx_target
    
    buttons.append(Button(
        lucide_icon(BROWSER_ICONS["refresh"], size=icons.icon_button),
        **refresh_attrs
    ))
    
    return Div(
        *buttons,
        cls=combine_classes(flex_display, gap(1))
    )

In [ ]:
# Test nav buttons
nav_btns = render_nav_buttons("/home", "/home/user", "/navigate")
html = to_xml(nav_btns)
assert "<button" in html.lower()
assert "<svg" in html.lower()  # Icons

# Test with no parent (at root)
nav_btns_root = render_nav_buttons(None, "/home/user", "/navigate")
html = to_xml(nav_btns_root)
assert "disabled" in html  # Parent button disabled

print("Nav buttons tests passed!")

Nav buttons tests passed!


## Path Bar

Complete path bar component combining breadcrumbs, buttons, and optional input.

In [ ]:
#| export
def render_path_bar(
    current_path: str,                      # Current directory path
    parent_path: Optional[str],             # Parent directory path
    home_path: str,                         # Home directory path
    config: FileBrowserConfig,              # Browser configuration
    navigate_url: str,                      # URL for navigation
    refresh_url: Optional[str] = None,      # URL for refresh
    hx_target: Optional[str] = None,        # HTMX target for swaps
    path_bar_id: Optional[str] = None,      # HTML ID for path bar
) -> Any:  # Path bar component
    """Render the complete path bar."""
    children = []
    
    # Navigation buttons
    children.append(render_nav_buttons(
        parent_path, home_path, navigate_url, refresh_url, hx_target
    ))
    
    # Breadcrumbs or path input
    if config.show_path_input:
        children.append(Div(
            render_path_input(current_path, navigate_url, hx_target),
            cls=combine_classes(grow(), min_w._0)
        ))
    elif config.show_breadcrumbs:
        children.append(Div(
            render_breadcrumbs(current_path, navigate_url, hx_target),
            cls=combine_classes(grow(), min_w._0, overflow.x.auto)
        ))
    else:
        # Just show path as text
        children.append(Div(
            Span(current_path, cls=combine_classes(font_size.sm, font_family.mono, truncate)),
            cls=combine_classes(grow(), min_w._0)
        ))
    
    # Build path bar
    bar_attrs = {
        "cls": combine_classes(
            flex_display, items.center, gap(2),
            p.x(2), p.y(1),
            bg_dui.base_200.opacity(50),
            border_dui.base_300, border.b()
        )
    }
    if path_bar_id:
        bar_attrs["id"] = path_bar_id
    
    return Div(*children, **bar_attrs)

In [ ]:
# Test complete path bar
config = FileBrowserConfig()
path_bar = render_path_bar(
    current_path="/home/user/Documents",
    parent_path="/home/user",
    home_path="/home/user",
    config=config,
    navigate_url="/navigate"
)
html = to_xml(path_bar)
assert "breadcrumbs" in html  # Default shows breadcrumbs
assert "Documents" in html
assert "<button" in html.lower()  # Nav buttons

# Test with path input
config_input = FileBrowserConfig(show_path_input=True, show_breadcrumbs=False)
path_bar = render_path_bar(
    current_path="/home/user",
    parent_path="/home",
    home_path="/home/user",
    config=config_input,
    navigate_url="/navigate"
)
html = to_xml(path_bar)
assert "<input" in html.lower()  # Path input shown
assert "<form" in html.lower()

print("Path bar tests passed!")

Path bar tests passed!


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()